# Beyond Constant Volatility: A First Look at the Heston Model

**Project type:** semi-learning paper notebook  
**Research question:** what changes when volatility is modelled as a stochastic process instead of a constant input?

## Abstract

This note introduces the Heston stochastic volatility model as a natural extension of Black-Scholes. Instead of assuming a constant volatility, Heston models variance as a mean-reverting stochastic process that can be correlated with the underlying asset. The notebook uses Monte Carlo simulation to build intuition about variance paths, the leverage effect, and option prices under stochastic volatility. The goal is not full calibration, but a first practical understanding of why stochastic volatility models exist.

## Personal Learning Position

After learning Black-Scholes, the biggest weakness that stands out to me is constant volatility. Implied-volatility smiles suggest that the market does not price every strike and maturity with one volatility number. Heston feels like a serious next step because it keeps the model structured but allows volatility itself to move. I am treating this notebook as a first exploration, not as a complete professional calibration exercise.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists():
        sys.path.insert(0, str(candidate / "src"))
        PROJECT_ROOT = candidate
        break

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from options_pricing_research.black_scholes import black_scholes_price
from options_pricing_research.heston import HestonParameters, feller_condition, heston_monte_carlo_price, simulate_heston_paths

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:.4f}".format

## 1. Model Setup

Under the Heston model, the stock price and variance follow:

$$
dS_t = (r-q)S_tdt + \sqrt{v_t}S_tdW_t^S,
$$

$$
dv_t = \kappa(\theta - v_t)dt + \xi\sqrt{v_t}dW_t^v,
$$

where $v_t$ is variance, $\kappa$ is mean reversion speed, $\theta$ is long-run variance, $\xi$ is volatility of variance, and the Brownian motions have correlation $\rho$.

The parameter I find most intuitive is $\rho$. If $\rho$ is negative, downward stock moves tend to coincide with rising variance. This resembles the leverage effect often discussed in equity markets.

In [ ]:
spot = 100.0
strike = 100.0
time_to_expiry = 1.0
risk_free_rate = 0.03

params = HestonParameters(
    initial_variance=0.04,
    long_run_variance=0.04,
    mean_reversion=2.0,
    vol_of_variance=0.35,
    correlation=-0.6,
)

pd.Series(
    {
        "initial_volatility": np.sqrt(params.initial_variance),
        "long_run_volatility": np.sqrt(params.long_run_variance),
        "mean_reversion": params.mean_reversion,
        "vol_of_variance": params.vol_of_variance,
        "correlation": params.correlation,
        "feller_condition_value": feller_condition(params),
    }
)

## 2. Simulated Spot and Volatility Paths

The next section simulates a small number of paths for visual intuition. This is not for pricing accuracy; it is for seeing how stochastic volatility behaves.

In [ ]:
demo_spots, demo_variances = simulate_heston_paths(
    spot,
    time_to_expiry,
    risk_free_rate,
    dividend_yield=0.0,
    params=params,
    paths=8,
    steps=126,
    seed=9,
)
demo_vols = np.sqrt(demo_variances)
time_grid = np.linspace(0, time_to_expiry, demo_spots.shape[1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i in range(demo_spots.shape[0]):
    axes[0].plot(time_grid, demo_spots[i], linewidth=1.2, alpha=0.85)
    axes[1].plot(time_grid, demo_vols[i], linewidth=1.2, alpha=0.85)

axes[0].set_title("Simulated Heston Spot Paths")
axes[0].set_xlabel("Years")
axes[0].set_ylabel("Spot")
axes[1].set_title("Simulated Volatility Paths")
axes[1].set_xlabel("Years")
axes[1].set_ylabel("Volatility")
fig.tight_layout()
plt.show()

The volatility paths make the model feel different from Black-Scholes immediately. Volatility is no longer just a number chosen before pricing. It is part of the state of the market. My learning takeaway is that stochastic volatility models try to explain why option prices can reflect changing uncertainty across time and market states.

## 3. Monte Carlo Pricing

The next comparison prices a European call under Heston and compares it with a Black-Scholes call using the same initial volatility. This is only an illustrative comparison, because a serious Heston paper would calibrate parameters to option-chain data.

In [ ]:
heston_price = heston_monte_carlo_price(
    "call",
    spot,
    strike,
    time_to_expiry,
    risk_free_rate,
    params,
    paths=40_000,
    steps=126,
    seed=21,
)
bs_price = black_scholes_price("call", spot, strike, time_to_expiry, risk_free_rate, np.sqrt(params.initial_variance))

pd.Series(
    {
        "black_scholes_price_initial_vol": bs_price,
        "heston_monte_carlo_price": heston_price.price,
        "heston_standard_error": heston_price.standard_error,
        "average_terminal_variance": heston_price.average_terminal_variance,
    }
)

I would be careful not to over-interpret this single price comparison. The important point is not that Heston is automatically better. The important point is that Heston gives the model a new degree of freedom: volatility can move randomly and be correlated with spot returns. That is the bridge toward explaining smiles and skews.

## 4. Reflection

My current view is that Heston is a model worth learning after Black-Scholes because it keeps the same no-arbitrage spirit but relaxes the constant-volatility assumption. It is also a good reminder that more advanced models bring more parameters. More realism can mean more calibration risk. For a student project, the next step should be to estimate or calibrate parameters from option-chain data instead of choosing them manually.

## References

- Heston, S. L. (1993). A Closed-Form Solution for Options with Stochastic Volatility with Applications to Bond and Currency Options. *Review of Financial Studies*.
- Gatheral, J. *The Volatility Surface: A Practitioner's Guide*.
- Hull, J. C. *Options, Futures, and Other Derivatives*.